# B7 U06 SL / TP — Implementation Freeze only
All switches OFF. Formal execution requires a separately reviewed commit and Chat approval. Fixed Stage1 72 only; no replacements. No U07 or Validation/Monitor.


In [ ]:
from pathlib import Path
RUN_MOUNT = False
RUN_INSTALL = False
RUN_CHECKOUT = False
RUN_INPUT_AUDIT = False
RUN_TESTS = False
RUN_SMOKE = False
RUN_PREFLIGHT = False
RUN_FORMAL = False
RUN_RESUME = False
RUN_ARCHIVE = False
IMPLEMENTATION_SHA = ""
APPROVAL = ""
REPO = Path("/content/time-entry-portfolio-lab-u06")
DATA_ROOT = Path("/content/drive/MyDrive/SET_FROZEN_M1_ROOT")
SELECTED = Path("/content/drive/MyDrive/b7_stage1_finalize_20261006/selected_top8.json")
STAGE1_CHECKPOINT = Path("/content/drive/MyDrive/b7_stage1_checkpoint_20261006")
LOCAL_ROOT = Path("/content/b7_u06_run")
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/SET_NEW_B7_U06_RUN_ID_checkpoints")
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/SET_NEW_B7_U06_RUN_ID_archive")


In [ ]:
if RUN_MOUNT:
    from google.colab import drive
    drive.mount("/content/drive")
if RUN_INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable,"-m","pip","install","numpy==2.3.5","pandas==2.2.3"])
    print("Restart runtime after installation before continuing")
if RUN_CHECKOUT:
    import subprocess
    if len(IMPLEMENTATION_SHA)!=40 or any(c not in "0123456789abcdef" for c in IMPLEMENTATION_SHA):
        raise ValueError("Set exact reviewed U06 implementation SHA")
    if REPO.exists(): raise FileExistsError("Fresh checkout required")
    subprocess.check_call(["git","clone","https://github.com/TR-KJ/time-entry-portfolio-lab.git",str(REPO)])
    subprocess.check_call(["git","-C",str(REPO),"checkout","--detach",IMPLEMENTATION_SHA])


In [ ]:
if any((RUN_INPUT_AUDIT,RUN_TESTS,RUN_SMOKE,RUN_PREFLIGHT,RUN_FORMAL,RUN_RESUME,RUN_ARCHIVE)):
    import sys
    sys.dont_write_bytecode=True
    sys.path.insert(0,str(REPO/"src/research"))
    from b7.u06_runtime import preflight,run_formal,archive_completed,input_config
    from b7.u06_input import extract
    from b7.stage1_runtime import run_tests
    from b7.stage1_input import resolve_exact
    from b7.u06_smoke import run_smoke
    if any((RUN_SMOKE,RUN_PREFLIGHT,RUN_FORMAL,RUN_RESUME)):
        paths=resolve_exact(DATA_ROOT)
if RUN_INPUT_AUDIT:
    audited=extract(SELECTED,STAGE1_CHECKPOINT)
    if audited!=input_config()[1]:raise ValueError("Compact input mismatch")
    print("Stage1 input audit PASS: 72")
if RUN_TESTS:print(run_tests()["Status"])
if RUN_SMOKE:print(run_smoke(paths))
if RUN_PREFLIGHT:
    proof=preflight(paths,SELECTED,STAGE1_CHECKPOINT,IMPLEMENTATION_SHA)
    print(proof["Status"])


In [ ]:
if RUN_FORMAL and RUN_RESUME:raise ValueError("Choose fresh run OR resume")
if RUN_FORMAL or RUN_RESUME:
    review=run_formal(paths,SELECTED,STAGE1_CHECKPOINT,IMPLEMENTATION_SHA,
        LOCAL_ROOT,DRIVE_CHECKPOINT_ROOT,approval=APPROVAL,resume=RUN_RESUME)
    print(review)
if RUN_ARCHIVE:
    print(archive_completed(LOCAL_ROOT/"final",DRIVE_ARCHIVE))


Dedicated approval: `CHAT_APPROVED_COLAB_B7_U06_ONLY`. Choose a new result-independent run-id before a fresh run. Drive mirroring is synchronous: each complete job is copied and verified, with marker published last, before the next job. Resume uses the same Drive root, exact implementation/config/input/data/environment and fresh local root after runtime loss. Python patch differences are rejected. U06 finalize-only is not implemented. Incomplete staging is never completed evidence; an existing malformed published job is rejected. Outputs expose candidate selections only after all 72 jobs complete. A failed finalization can be retried via a fresh local root restored from the same exact completed Drive jobs.
